# 뉴스 데이터 수집

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import pytz
now_kst = datetime.now(pytz.timezone('Asia/Seoul'))
week_ago = now_kst - timedelta(days=4)
def get_yahoo_news_links(ticker="AAPL", limit=10):
    url = f"https://finance.yahoo.com/quote/{ticker}/news"
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = []
    for li in soup.select("li.stream-item.story-item")[:limit]:  # 🔥 20개만
        try:
            title_tag = li.select_one("h3")
            link_tag = li.select_one("a")
            if not title_tag or not link_tag:
                continue

            title = title_tag.get_text(strip=True)
            link = link_tag["href"]
            if not link.startswith("http"):
                link = "https://finance.yahoo.com" + link

            articles.append({
                "회사": ticker,
                "제목": title,
                "URL": link
            })
        except Exception as e:
            print("❌ 뉴스 항목 파싱 오류:", e)
    return articles


def extract_article_body(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers, timeout=20)
    soup = BeautifulSoup(res.text, "html.parser")

    # 🔍 본문 추출
    paragraphs = soup.select("article p")
    if not paragraphs:
        paragraphs = soup.select("div.caas-body p")

    body_text = "\n".join(p.get_text(strip=True) for p in paragraphs if p.text.strip())

    # 🔍 날짜 추출
    time_tag = soup.find("time", {"class": "byline-attr-meta-time"})
    if time_tag and time_tag.has_attr("datetime"):
        utc_time = datetime.strptime(time_tag["datetime"], "%Y-%m-%dT%H:%M:%S.000Z")
        kst_time = utc_time.replace(tzinfo=pytz.UTC).astimezone(pytz.timezone("Asia/Seoul"))
        date_str = kst_time.strftime("%Y-%m-%d %H:%M")
    else:
        date_str = "날짜 없음"

    return body_text, date_str
# 위키피디아의 S&P 500 구성 종목 목록 URL
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# HTML 테이블을 읽어오기
tables = pd.read_html(url)
df = tables[0]  # 첫 번째 테이블이 S&P 500 구성 종목 목록


# 티커와 기업명 추출
ticker_Group= df['Symbol'].tolist()

tickers = [t.replace(".", "-") for t in ticker_Group]


df_list = []
for ticker in tickers:
    print(f"{ticker} 뉴스 수집 중...")
    articles = get_yahoo_news_links(ticker)

    if not articles:
        articles = [{
            "회사": ticker,
            "제목": "뉴스 없음",
            "URL": "",
            "본문": "",
            "날짜": "날짜 없음",
            "요약": "요약 실패: 뉴스 없음",
            "감정": "감정 분석 실패",
            "감정확신도": 0.0
        }]
    else:
        filtered_articles = []
        for article in articles:
            try:
                body, date_str = extract_article_body(article["URL"])
                article["본문"] = body
                article["날짜"] = date_str

                # 날짜 필터링
                news_time = datetime.strptime(date_str, "%Y-%m-%d %H:%M")
                news_time = pytz.timezone("Asia/Seoul").localize(news_time)
                if news_time >= week_ago:
                    filtered_articles.append(article)
            except Exception as e:
                article["본문"] = f"본문 추출 실패: {e}"
                article["날짜"] = "날짜 추출 실패"

        articles = filtered_articles

        # 📌 필터링 후에도 비었으면 기본값 추가
        if not articles:
            articles = [{
                "회사": ticker,
                "제목": "최근 7일 내 뉴스 없음",
                "URL": "",
                "본문": "",
                "날짜": "날짜 없음",
                "요약": "요약 실패",
                "감정": "감정 분석 실패",
                "감정확신도": 0.0
            }]

    # DataFrame 저장
    df = pd.DataFrame(articles)
    df_list.append(df)

# ✅ 전체 뉴스 통합
all_news_df = pd.concat(df_list, ignore_index=True)

# ✅ 결과 확인
print(all_news_df[["회사", "제목", "날짜"]].head())


MMM 뉴스 수집 중...
AOS 뉴스 수집 중...
ABT 뉴스 수집 중...
ABBV 뉴스 수집 중...
ACN 뉴스 수집 중...
ADBE 뉴스 수집 중...
AMD 뉴스 수집 중...
AES 뉴스 수집 중...
AFL 뉴스 수집 중...
A 뉴스 수집 중...
APD 뉴스 수집 중...
ABNB 뉴스 수집 중...
AKAM 뉴스 수집 중...
ALB 뉴스 수집 중...
ARE 뉴스 수집 중...
ALGN 뉴스 수집 중...
ALLE 뉴스 수집 중...
LNT 뉴스 수집 중...
ALL 뉴스 수집 중...
GOOGL 뉴스 수집 중...
GOOG 뉴스 수집 중...
MO 뉴스 수집 중...
AMZN 뉴스 수집 중...
AMCR 뉴스 수집 중...
AEE 뉴스 수집 중...
AEP 뉴스 수집 중...
AXP 뉴스 수집 중...
AIG 뉴스 수집 중...
AMT 뉴스 수집 중...
AWK 뉴스 수집 중...
AMP 뉴스 수집 중...
AME 뉴스 수집 중...
AMGN 뉴스 수집 중...
APH 뉴스 수집 중...
ADI 뉴스 수집 중...
ANSS 뉴스 수집 중...
AON 뉴스 수집 중...
APA 뉴스 수집 중...
APO 뉴스 수집 중...
AAPL 뉴스 수집 중...
AMAT 뉴스 수집 중...
APTV 뉴스 수집 중...
ACGL 뉴스 수집 중...
ADM 뉴스 수집 중...
ANET 뉴스 수집 중...
AJG 뉴스 수집 중...
AIZ 뉴스 수집 중...
T 뉴스 수집 중...
ATO 뉴스 수집 중...
ADSK 뉴스 수집 중...
ADP 뉴스 수집 중...
AZO 뉴스 수집 중...
AVB 뉴스 수집 중...
AVY 뉴스 수집 중...
AXON 뉴스 수집 중...
BKR 뉴스 수집 중...
BALL 뉴스 수집 중...
BAC 뉴스 수집 중...
BAX 뉴스 수집 중...
BDX 뉴스 수집 중...
BRK-B 뉴스 수집 중...
BBY 뉴스 수집 중...
TECH 뉴스 수집 중...
BIIB 뉴스 수집 중...
BLK 뉴스 수집 중...
BX 뉴스

# 뉴스 요약

In [2]:
all_news_df1 = all_news_df[["회사", "제목", "날짜","본문"]]
all_news_df1

,회사,제목,날짜,본문
0,MMM,Driving Personal Growth: 3M's Wendy Bauer on N...,2025-06-03 23:00,"NORTHAMPTON, MA /ACCESS Newswire/ June 3, 2025..."
1,AOS,최근 7일 내 뉴스 없음,날짜 없음,
2,ABT,Media Groups Probed by FTC Over Coordinating B...,2025-06-03 23:30,(Bloomberg) -- The US Federal Trade Commission...
3,ABT,There's Been No Shortage Of Growth Recently Fo...,2025-06-01 20:00,"If you're looking for a multi-bagger, there's ..."
4,ABBV,ASCO25: AbbVie flexes ADC Phase I data as NSCL...,2025-06-04 01:09,AbbVie’s antibody drug conjugate (ADC) Temab-A...
...,...,...,...,...
577,XYL,뉴스 없음,날짜 없음,
578,YUM,뉴스 없음,날짜 없음,
579,ZBRA,뉴스 없음,날짜 없음,
580,ZBH,뉴스 없음,날짜 없음,


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import pandas as pd

# ✅ 모델 로딩
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, use_safetensors=True)
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

# ✅ 긴 본문 자동 분할 요약 함수
def summarize_text(text, max_chunk_tokens=900):
    inputs = tokenizer(text, return_tensors="pt", truncation=False)
    input_ids = inputs["input_ids"][0]
    total_tokens = len(input_ids)

    if total_tokens <= max_chunk_tokens:
        return summarizer(text, max_length=100, min_length=20, do_sample=False)[0]["summary_text"]

    summaries = []
    for i in range(0, total_tokens, max_chunk_tokens):
        chunk_ids = input_ids[i:i + max_chunk_tokens]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        summary = summarizer(chunk_text, max_length=80, min_length=15, do_sample=False)[0]["summary_text"]
        summaries.append(summary)

    return " ".join(summaries)

# ✅ DataFrame 불러오기 및 요약 칼럼 추가
# df = pd.read_csv("your_file.csv")  # 필요 시 주석 해제
summaries = []

for i, text in enumerate(all_news_df1["본문"]):
    try:
        if not isinstance(text, str) or text.strip() == "" or "뉴스 없음" in text or "본문 추출 실패" in text:
            summary = "뉴스 없음"
        else:
            summary = summarize_text(text)
    except Exception as e:
        print(f"[{i}번째 요약 실패] {e}")
        summary = "[요약 실패]"
    summaries.append(summary)

all_news_df1["요약"] = summaries



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Your max_length is set to 100, but your input_length is only 23. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=11)
Your max_length is set to 100, but your input_length is only 25. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)
Your max_length is set to 100, but your input_length is only 25. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)
Your max_length is set to 100, but your input_length is only 25. Since this is a summarization task, where outputs shor

# 코랩 시

In [4]:
import os
import pandas as pd
from datetime import date, datetime
from google.colab import drive
import pytz
# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 오늘 날짜
seoul_tz = pytz.timezone("Asia/Seoul")
today = datetime.now(seoul_tz).strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/뉴스_요약데이터/({today})뉴스_요약.csv"

all_news_df1.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

Mounted at /content/drive
✅ 저장 완료: /content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/뉴스_요약데이터/(2025-06-04)뉴스_요약.csv


# 로컬 시

In [ ]:
seoul_tz = pytz.timezone("Asia/Seoul")
today = datetime.now(seoul_tz).strftime("%Y-%m-%d")
file_path = f"/Users/kjb/Desktop/hateslop/프로젝트/뉴스_요약데이터/({today})뉴스_요약.csv"

all_news_df1.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

# 뉴스 감정 분석

In [ ]:
import pandas as pd
import re
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

df1 = all_news_df1.copy()
nlp = spacy.load("en_core_web_sm")

from transformers import pipeline as hf_pipeline
ner = hf_pipeline("ner", model="dslim/bert-base-NER", tokenizer="dslim/bert-base-NER", aggregation_strategy="simple")

model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True)
sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

from collections import Counter

sentiment_results = []

for _, row in df1.iterrows():
    try:
        text = row["요약"]
        company_name = row["회사"]

        # 뉴스 없음 또는 요약 실패일 경우 Neutral 처리
        if not isinstance(text, str) or text.strip() == "" or "뉴스 없음" in text or "요약 실패" in text:
            sentiment_results.append("Neutral")
            continue

        # 1. 문장 분리
        split_points = [m.start() for m in re.finditer(r'\bwhile\b|\bbut\b', text)]
        clauses = []
        start = 0
        for point in split_points:
            clauses.append(text[start:point].strip())
            start = point
        clauses.append(text[start:].strip())

        # 2. 병합
        final_clauses = []
        for i, clause in enumerate(clauses):
            doc = nlp(clause)
            if i > 0 and len(doc) > 0 and doc[0].pos_ == "VERB":
                final_clauses[-1] += " " + clause
            else:
                final_clauses.append(clause)

        # 3. 감성 분석 (회사명 언급된 절 모두)
        company_sentiments = []
        for clause in final_clauses:
            ents = ner(clause)
            orgs = [e["word"] for e in ents if e["entity_group"] == "ORG"]

            if not orgs and company_name in clause:
                orgs.append(company_name)

            if any(company_name in o for o in orgs):
                preds = sentiment(clause)[0]
                top = max(preds, key=lambda x: x["score"])
                company_sentiments.append(top["label"])

        # 4. 다수결 또는 fallback
        if company_sentiments:
            final_sentiment = Counter(company_sentiments).most_common(1)[0][0]
            sentiment_results.append(final_sentiment)
        else:
            preds = sentiment(text[:512])[0]
            top = max(preds, key=lambda x: x["score"])
            sentiment_results.append(top["label"])

    except Exception as e:
        print(f"❌ 예외 발생: {e}")
        sentiment_results.append("Neutral")
df1["감성분석"] = sentiment_results

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnin

In [5]:
import pandas as pd
import re
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

df1 = all_news_df1.copy()
# 2. spaCy 로드
nlp = spacy.load("en_core_web_sm")

# 3. NER 파이프라인
from transformers import pipeline as hf_pipeline
ner = hf_pipeline("ner", model="dslim/bert-base-NER", tokenizer="dslim/bert-base-NER", aggregation_strategy="simple")

# 4. 감성 분석 파이프라인
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True)
sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, return_all_scores=True)

# 5. 결과 저장용 리스트
sentiment_results = []

for _, row in df1.iterrows():
    try:
        text = row["요약"]
        company_name = row["회사"]

        # 🔒 뉴스 없음 또는 요약 실패일 경우 Neutral 처리
        if not isinstance(text, str) or text.strip() == "" or "뉴스 없음" in text or "요약 실패" in text:
            sentiment_results.append("Neutral")
            continue

        # 1. 문장 분리
        split_points = [m.start() for m in re.finditer(r'\bwhile\b|\bbut\b', text)]
        clauses = []
        start = 0
        for point in split_points:
            clauses.append(text[start:point].strip())
            start = point
        clauses.append(text[start:].strip())

        # 2. 병합
        final_clauses = []
        for i, clause in enumerate(clauses):
            doc = nlp(clause)
            if i > 0 and len(doc) > 0 and doc[0].pos_ == "VERB":
                final_clauses[-1] += " " + clause
            else:
                final_clauses.append(clause)

        # 3. 감성 분석
        found = False
        for clause in final_clauses:
            ents = ner(clause)
            orgs = [e["word"] for e in ents if e["entity_group"] == "ORG"]

            if not orgs and company_name in clause:
                orgs.append(company_name)

            if any(company_name in o for o in orgs):
                preds = sentiment(clause)[0]
                top = max(preds, key=lambda x: x["score"])
                sentiment_results.append(top["label"])
                found = True
                break

        # 4. Fallback: 전체 텍스트로 분석
        if not found:
            preds = sentiment(text[:512])[0]
            top = max(preds, key=lambda x: x["score"])
            sentiment_results.append(top["label"])

    except Exception as e:
        print(f"❌ 예외 발생: {e}")
        sentiment_results.append("Neutral")

# ✅ 결과 저장
df1["감성분석"] = sentiment_results

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

...

Device set to use cuda:0


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [14]:

df1

,회사,제목,날짜,본문,요약,감성분석
0,MMM,Driving Personal Growth: 3M's Wendy Bauer on N...,2025-06-03 23:00,"NORTHAMPTON, MA /ACCESS Newswire/ June 3, 2025...","Wendy Bauer, Group President of 3M's Transport...",Positive
1,AOS,최근 7일 내 뉴스 없음,날짜 없음,,뉴스 없음,Neutral
2,ABT,Media Groups Probed by FTC Over Coordinating B...,2025-06-03 23:30,(Bloomberg) -- The US Federal Trade Commission...,The FTC is investigating whether more than a d...,Neutral
3,ABT,There's Been No Shortage Of Growth Recently Fo...,2025-06-01 20:00,"If you're looking for a multi-bagger, there's ...",Abbott Laboratories(NYSE:ABT) has an ROCE of 1...,Positive
4,ABBV,ASCO25: AbbVie flexes ADC Phase I data as NSCL...,2025-06-04 01:09,AbbVie’s antibody drug conjugate (ADC) Temab-A...,"AbbVie’s candidate, dubbed internally as ABBV-...",Positive
...,...,...,...,...,...,...
577,XYL,뉴스 없음,날짜 없음,,뉴스 없음,Neutral
578,YUM,뉴스 없음,날짜 없음,,뉴스 없음,Neutral
579,ZBRA,뉴스 없음,날짜 없음,,뉴스 없음,Neutral
580,ZBH,뉴스 없음,날짜 없음,,뉴스 없음,Neutral


# 감정 점수 계산

In [15]:
# 감정 점수 계산 함수 (가중치 × 확신도)
# 감정 점수 매핑
finbert_score_map = {
    "Positive": 1,
    "Negative": -1,
    "Neutral": 0
}

# 감성 점수를 숫자로 변환
df1["감성점수"] = df1["감성분석"].map(finbert_score_map)

# 티커별 평균 점수 계산
mean_scores = df1.groupby("회사")["감성점수"].mean().reset_index()
mean_scores.columns = ["회사","감정점수"]


In [16]:
mean_scores

,회사,감정점수
0,A,0.000000
1,AAPL,0.000000
2,ABBV,0.500000
3,ABNB,-0.333333
4,ABT,0.500000
...,...,...
498,XYL,0.000000
499,YUM,0.000000
500,ZBH,0.000000
501,ZBRA,0.000000


In [17]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# HTML 테이블을 읽어오기
tables = pd.read_html(url)
df = tables[0]  # 첫 번째 테이블이 S&P 500 구성 종목 목록


# 티커와 기업명 추출
ticker_Group= df['Symbol'].tolist()
company_names = df['Security'].tolist()

ticker_Group = [t.replace(".", "-") for t in ticker_Group]
company_names = [t.replace(".", "-") for t in company_names]
ticker_names = dict(zip(ticker_Group, company_names))

# 안전한 매핑: 없으면 원래 값 유지
mean_scores["종목"] = mean_scores["회사"].map(ticker_names)
mean_scores.drop('회사', inplace=True, axis = 1)


In [18]:
mean_scores

,감정점수,종목
0,0.000000,Agilent Technologies
1,0.000000,Apple Inc-
2,0.500000,AbbVie
3,-0.333333,Airbnb
4,0.500000,Abbott Laboratories
...,...,...
498,0.000000,Xylem Inc-
499,0.000000,Yum! Brands
500,0.000000,Zimmer Biomet
501,0.000000,Zebra Technologies


In [19]:
import os
import pandas as pd
from datetime import date, datetime
from google.colab import drive
import pytz

# 2. 오늘 날짜
seoul_tz = pytz.timezone("Asia/Seoul")
today = datetime.now(seoul_tz).strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/뉴스데이터/({today})뉴스.csv"

mean_scores.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

✅ 저장 완료: /content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/뉴스데이터/(2025-06-04)뉴스.csv
